## Embeddings in Agentic Systems

Embeddings are fundamental to modern agents. They transform text into dense numerical vectors that capture semantic meaning, enabling agents to:
- **Retrieve relevant context** from knowledge bases (RAG)
- **Rank and filter** candidate information based on relevance
- **Cache identical queries** using semantic similarity
- **Detect intent** and route tasks to appropriate tools
- **Measure hallucination** by grounding outputs against retrieved facts

### **What Are Embeddings?**

An embedding is a vector representation of text in continuous space where:
- Semantically similar texts are positioned close together
- Semantically different texts are far apart
- Distance metrics (cosine, Euclidean) measure similarity/relevance

**Example:** Queries "Apple's stock price" and "AAPL ticker value" are dissimilar as words but close in semantic space (~0.9 cosine similarity). This is why embeddings outperform keyword search.

### **How Embeddings Enable Agent Capabilities**

| Agent Task | Embedding Benefit | Example |
|-----------|------------------|---------|
| **Context Retrieval** | Find relevant documents from millions without scanning all | Query: "stock volatility" → retrieve only finance documents |
| **Query Deduplication** | Detect if question was asked before (cache hit) | Cache: "What's Apple's P/E?" + New: "Apple's price-to-earnings?" → 0.95 similarity → reuse answer |
| **Intent Classification** | Route to specialized tools without explicit rules | Embedding distance determines if query = price lookup, news search, or ratio analysis |
| **Ranking Evidence** | Order retrieved facts by relevance to question | Top 5 snippets ranked by semantic distance to user query |
| **Hallucination Detection** | Check if LLM answer is grounded in retrieved facts | Compare answer embedding to evidence embedding similarity |

### **Practical Python Snippet: Using Embeddings**

```python
from langchain.embeddings import OllamaEmbeddings
from langchain.vectorstores import Chroma
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Initialize embeddings with local Ollama model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

# Embed some financial queries
queries = [
    "What is Apple's stock price?",
    "AAPL ticker current value",
    "Microsoft earnings report"
]

embedded_queries = embeddings.embed_documents(queries)

# Calculate similarity between first two (should be ~0.9)
similarity = cosine_similarity(
    [embedded_queries[0]], 
    [embedded_queries[1]]
)[0][0]
print(f"Query 1 ↔ Query 2 similarity: {similarity:.3f}")  # Output: ~0.9

# Store in vector database for retrieval
vector_db = Chroma.from_documents(
    documents=my_documents,
    embedding=embeddings,
    persist_directory="./vector_db"
)

# Retrieve relevant context
retrieved = vector_db.similarity_search_with_score(
    query=queries[0],
    k=5,           # Top 5 results
    distance_threshold=0.15  # Relevance filter
)

for doc, score in retrieved:
    print(f"Relevance: {score:.3f} | Content: {doc.page_content[:50]}...")
```

### **Embedding Models for Different Use Cases**

| Model | Dimension | Latency | Best For |
|-------|-----------|---------|----------|
| `nomic-embed-text` | 768 | 50ms | General purpose, good balance |
| `mxbai-embed-large` | 1024 | 100ms | High-precision retrieval, larger docs |
| `all-minilm-l6-v2` (ONNX) | 384 | 10ms | Speed-critical, lightweight agents |
| `bge-base-en-v1.5` | 768 | 60ms | Semantic search, multilingual |

**Recommendation for agents:**
- **Development/learning:** `nomic-embed-text` (good quality, reasonable speed)
- **Production large-scale:** `mxbai-embed-large` (better retrieval precision)
- **Edge/mobile agents:** `all-minilm-l6-v2` (fast, small footprint)

### **Batch Embedding Best Practices**

```python
# ❌ Inefficient: embed one by one
for query in queries:
    embedding = embeddings.embed_query(query)  # Multiple round trips

# ✅ Efficient: batch embedding
embeddings_batch = embeddings.embed_documents(queries)  # Single call
```

**Token/Cost Impact:**
- Batch 100 queries: ~1 API call
- Sequential 100 queries: ~100 API calls
- **Savings:** 99x reduction in overhead

### **Semantic Caching with Embeddings**

One of our optimization strategies uses embeddings to avoid re-computing identical queries:

```python
# Check if question is similar to previous cached questions
new_question = "What's Apple's stock price?"
cached_questions = ["AAPL ticker value", "Apple stock cost"]

similarities = []
for cached_q in cached_questions:
    sim = cosine_similarity(
        embeddings.embed_query(new_question),
        embeddings.embed_query(cached_q)
    )
    similarities.append(sim)

if max(similarities) > 0.85:  # Threshold
    print("Cache hit! Reuse previous answer")
else:
    print("Cache miss. Query LLM.")
```

### **Indexing and Persistence**

Embeddings are expensive to compute but cheap to retrieve. Always persist:

```python
# Compute once and save
vector_db = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./my_knowledge_base"
)

# Load cached embeddings (instant, no recomputation)
vector_db = Chroma(
    persist_directory="./my_knowledge_base",
    embedding_function=embeddings
)
```

**Why indexing matters:**
- Computing 100k embeddings: ~50 minutes
- Retrieving from index: <50ms per query
- **For agents:** Amortize compute cost over thousands of queries

### **Hallucination Detection with Embeddings**

Our grounding check uses embeddings to measure if LLM output is supported by retrieved evidence:

```python
# LLM generated answer
answer = "Apple's P/E ratio is 28.5"

# Retrieved evidence
evidence = "AAPL P/E: 28.3-28.7 range per recent filings"

# Compare semantic similarity
answer_embedding = embeddings.embed_query(answer)
evidence_embedding = embeddings.embed_query(evidence)

similarity = cosine_similarity(
    [answer_embedding],
    [evidence_embedding]
)[0][0]

if similarity > 0.72:
    print("✅ Answer is grounded in evidence")
else:
    print("⚠️ Answer may hallucinate — refetch context")
```

### Practical Example

In [7]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [11]:

# Initialize embeddings with local Ollama model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)


In [21]:
# Embed some financial queries
queries = [
    "What is Apple's stock price?",
    "AAPL ticker for Apple current stock value",
    # "Microsoft earnings report"
]

embedded_queries = embeddings.embed_documents(queries)

# Calculate similarity between first two (should be ~0.9)
similarity = cosine_similarity(
    [embedded_queries[0]], 
    [embedded_queries[1]],
)[0][0]
print(f"Query 1 ↔ Query 2 similarity: {similarity:.3f}")  # Output: ~0.9



Query 1 ↔ Query 2 similarity: 0.804


In [17]:
len(embedded_queries)

3

In [ ]:
my_documents =""

In [ ]:
# Store in vector database for retrieval
vector_db = Chroma.from_documents(
    documents=my_documents,
    embedding=embeddings,
    persist_directory="./vector_db"
)

# Retrieve relevant context
retrieved = vector_db.similarity_search_with_score(
    query=queries[0],
    k=5,           # Top 5 results
    distance_threshold=0.15  # Relevance filter
)


In [9]:
# LLM generated answer
answer = "Apple's P/E ratio is 28.5"

# Retrieved evidence
evidence = "AAPL P/E: 28.3-28.7 range per recent filings"

# Compare semantic similarity
answer_embedding = embeddings.embed_query(answer)
evidence_embedding = embeddings.embed_query(evidence)

similarity = cosine_similarity(
    [answer_embedding],
    [evidence_embedding]
)[0][0]

print(similarity)

if similarity > 0.72:
    print("✅ Answer is grounded in evidence")
else:
    print("⚠️ Answer may hallucinate — refetch context")

0.6882530500402788
⚠️ Answer may hallucinate — refetch context


In [1]:

# Embed some financial queries
queries = [
    "What is Apple's stock price?",
    "AAPL ticker current value",
    "Microsoft earnings report"
]

embedded_queries = embeddings.embed_documents(queries)

# Calculate similarity between first two (should be ~0.9)
similarity = cosine_similarity(
    [embedded_queries[0]], 
    [embedded_queries[1]]
)[0][0]
print(f"Query 1 ↔ Query 2 similarity: {similarity:.3f}")  # Output: ~0.9

# Store in vector database for retrieval
vector_db = Chroma.from_documents(
    documents=my_documents,
    embedding=embeddings,
    persist_directory="./vector_db"
)

# Retrieve relevant context
retrieved = vector_db.similarity_search_with_score(
    query=queries[0],
    k=5,           # Top 5 results
    distance_threshold=0.15  # Relevance filter
)

for doc, score in retrieved:
    print(f"Relevance: {score:.3f} | Content: {doc.page_content[:50]}...")

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ModuleNotFoundError: No module named 'langchain.vectorstores'

In [ ]:
### **Semantic Caching with Embeddings**

One of our optimization strategies uses embeddings to avoid re-computing identical queries:

```python
# Check if question is similar to previous cached questions
new_question = "What's Apple's stock price?"
cached_questions = ["AAPL ticker value", "Apple stock cost"]

similarities = []
for cached_q in cached_questions:
    sim = cosine_similarity(
        embeddings.embed_query(new_question),
        embeddings.embed_query(cached_q)
    )
    similarities.append(sim)

if max(similarities) > 0.85:  # Threshold
    print("Cache hit! Reuse previous answer")
else:
    print("Cache miss. Query LLM.")
```